# Tutorial 6: Cluster Analysis

This tutorial demonstrates how to analyze stellar clusters using brutus, including isochrone fitting, binary modeling, and parameter estimation.

## Topics Covered

1. **Loading cluster data** (M67 example)
2. **Isochrone fitting** with `isochrone_population_loglike`
3. **Cluster parameters** (age, metallicity, distance, extinction)
4. **Photometric offsets** determination
5. **Binary fraction** modeling
6. **MCMC sampling** for uncertainties

## Prerequisites

This tutorial requires the following brutus data files:
- `MIST_1.2_iso_vvcrit0.0.h5` - MIST isochrones
- `nn_c3k.h5` - Neural network for bolometric corrections
- `NGC_2682.fits` - M67 cluster data

If you don't have these files, run the optional download cell below.

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tutorial_utils import (
    setup_tutorial,
    find_brutus_data_file,
    save_figure as _save_fig,
    print_section,
    load_m67_data,
)

info = setup_tutorial(6, title="Tutorial 06: Cluster Analysis")
plots_dir = info['plot_dir']


def save_figure(fig, name):
    """Save figure to this tutorial's plot directory."""
    _save_fig(fig, 6, name)

In [ ]:
from brutus.utils import inv_magnitude
from brutus.data import filters

# Load M67 data
print("Loading M67 cluster data...")
data_dict = load_m67_data()
fdata = data_dict['data']
Nobj = len(fdata)
print(f"  Loaded {Nobj} sources")

# ---- Define the combined filter set: Gaia + PS1(grizy) + 2MASS ----
# This gives 11 bands spanning optical through near-IR.
combined_filters = filters.gaia + filters.ps[:5] + filters.tmass
n_filters = len(combined_filters)
print(f"  Using {n_filters} filters: {combined_filters}")

# ---- Column names in the FITS catalogue ----
mag_columns = [
    # Gaia DR2 (revised passband)
    'Gaia_G_DR2Rev', 'Gaia_BP_DR2Rev', 'Gaia_RP_DR2Rev',
    # Pan-STARRS grizy
    'PS_g', 'PS_r', 'PS_i', 'PS_z', 'PS_y',
    # 2MASS JHKs
    '2MASS_J', '2MASS_H', '2MASS_Ks',
]
err_columns = [c + '_Err' for c in mag_columns]

# ---- Convert calibrated magnitudes to maggies ----
# brutus works in "maggie" units: flux = 10^(-0.4 * mag), with no
# zero-point offset.  inv_magnitude handles NaN gracefully.
all_mag = np.column_stack([fdata[c] for c in mag_columns])
all_magerr = np.column_stack([fdata[c] for c in err_columns])

phot, err = inv_magnitude(all_mag, all_magerr)

# Add a 2% systematic error floor (accounts for model systematics)
err = np.sqrt(err**2 + (0.02 * phot)**2)
mask = np.isfinite(phot) & (err > 0) & (phot > 0)

# Replace NaN/invalid values with safe placeholders for masked bands.
# The mask tells the likelihood to skip these, but phot_loglike needs
# finite values everywhere to avoid NaN propagation (NaN * 0 = NaN).
phot = np.where(mask, phot, 1.0)
err = np.where(mask, err, 1.0)

# Band availability summary
band_names = ['G', 'BP', 'RP', 'g_PS', 'r_PS', 'i_PS', 'z_PS', 'y_PS',
              'J', 'H', 'Ks']
print("\nBand availability (all sources):")
for i, name in enumerate(band_names):
    n_valid = np.sum(mask[:, i])
    print(f"  {name:>5s}: {n_valid:4d}/{Nobj} ({100*n_valid/Nobj:.0f}%)")

# ---- Parallax (with Gaia DR2 zero-point correction) ----
parallax = fdata['Parallax'] + 0.054          # Lindegren et al. 2018
parallax_err = np.sqrt(fdata['Parallax_Err']**2 + 0.043**2)

# ---- Membership probability ----
try:
    pmem = fdata['HDBscan_MemProb']
    print("\n  Found membership probabilities")
except Exception:
    pmem = np.ones(Nobj)
    print("\n  No membership info, assuming all are members")

# ---- Quality cuts ----
# Require: at least 3 valid bands, high membership, valid parallax
# No magnitude cut needed -- multiband data constrains all masses well.
gaia_g = all_mag[:, 0]
n_valid_bands = np.sum(mask, axis=1)
quality = (
    (pmem > 0.5)
    & (n_valid_bands >= 3)
    & np.isfinite(parallax)
)

# Store Gaia mags for CMD plotting later
gaia_mag = all_mag[:, :3]

print(f"\nHigh-confidence members: {quality.sum()} / {Nobj}")
print(f"  Median valid bands per star: {np.median(n_valid_bands[quality]):.0f}")
print(f"  Mean parallax: {np.nanmean(parallax[quality]):.3f} "
      f"+/- {np.nanstd(parallax[quality]):.3f} mas")
print(f"  Implied distance: {1000 / np.nanmean(parallax[quality]):.0f} pc")

In [ ]:
# Extract auxiliary columns for plotting
try:
    ra = fdata['gaia_dr2_source.ra']
    dec = fdata['gaia_dr2_source.dec']
except Exception:
    ra = np.full(Nobj, np.nan)
    dec = np.full(Nobj, np.nan)

# Package selected data for analysis
cluster_data = {
    'phot': phot[quality],
    'err': err[quality],
    'mask': mask[quality],
    'parallax': parallax[quality],
    'parallax_err': parallax_err[quality],
    'pmem': pmem[quality],
}

n_stars = len(cluster_data['phot'])
cluster_prob = np.mean(cluster_data['pmem'])

print(f"Data ready for analysis: {n_stars} stars, cluster_prob = {cluster_prob:.3f}")

In [ ]:
# Visualize cluster data
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Gaia magnitudes for CMD plotting (indices 0,1,2 = G, BP, RP)
bp_rp = gaia_mag[:, 1] - gaia_mag[:, 2]
g = gaia_mag[:, 0]

# Panel 1: Gaia CMD
ax = axes[0, 0]
scatter = ax.scatter(bp_rp[quality], g[quality],
                    c=pmem[quality], s=5, cmap='RdYlBu_r',
                    vmin=0, vmax=1, alpha=0.8)
ax.set_xlabel('BP - RP')
ax.set_ylabel('G')
ax.set_title('M67 Gaia CMD')
ax.invert_yaxis()
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(18, 8)
plt.colorbar(scatter, ax=ax, label='P(member)')
ax.grid(True, alpha=0.3)

# Panel 2: Spatial distribution
ax = axes[0, 1]
try:
    ra = fdata['gaia_dr2_source.ra']
    dec = fdata['gaia_dr2_source.dec']
except Exception:
    ra = np.full(Nobj, np.nan)
    dec = np.full(Nobj, np.nan)
scatter = ax.scatter(ra[quality], dec[quality], c=pmem[quality],
                    s=5, cmap='RdYlBu_r', vmin=0, vmax=1, alpha=0.8)
ax.set_xlabel('RA (deg)')
ax.set_ylabel('Dec (deg)')
ax.set_title('Spatial Distribution')
ax.set_aspect('equal')
plt.colorbar(scatter, ax=ax, label='P(member)')
ax.grid(True, alpha=0.3)

# Panel 3: Parallax distribution
ax = axes[0, 2]
valid_plx = quality & np.isfinite(parallax)
ax.hist(parallax[valid_plx], bins=30, alpha=0.7, color='blue', edgecolor='darkblue')
ax.axvline(1.11, color='red', ls='--', lw=2, label='Expected (900 pc)')
ax.axvline(np.median(parallax[valid_plx]), color='green', ls='--', lw=2,
           label=f'Median: {np.median(parallax[valid_plx]):.2f}')
ax.set_xlabel('Parallax (mas)')
ax.set_ylabel('Number of Stars')
ax.set_title('Parallax Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Band coverage per star
ax = axes[1, 0]
ax.hist(n_valid_bands[quality], bins=np.arange(0.5, n_filters + 1.5, 1),
        alpha=0.7, color='green', edgecolor='darkgreen')
ax.set_xlabel('Number of Valid Bands')
ax.set_ylabel('Number of Stars')
ax.set_title(f'Band Coverage ({n_filters} bands available)')
ax.grid(True, alpha=0.3)

# Panel 5: Color distribution
ax = axes[1, 1]
ax.hist(bp_rp[quality], bins=30, alpha=0.7, color='orange', edgecolor='darkorange')
ax.set_xlabel('BP - RP')
ax.set_ylabel('Number of Stars')
ax.set_title('Color Distribution')
ax.grid(True, alpha=0.3)

# Panel 6: Cluster properties
ax = axes[1, 2]
ax.axis('off')

info_text = f"""
M67 (NGC 2682) Properties:

Literature values:
  - Age: 3.5-4.0 Gyr
  - [Fe/H]: -0.05 to +0.05
  - Distance: 850-950 pc
  - E(B-V): 0.04-0.05
  - Binary fraction: 15-25%

Data summary:
  - Total sources: {Nobj}
  - High-prob members: {quality.sum()}
  - Bands: Gaia + PS1 + 2MASS ({n_filters})
  - Mean parallax: {np.nanmean(parallax[quality]):.3f} mas
  - Implied distance: {1000/np.nanmean(parallax[quality]):.0f} pc
"""

ax.text(0.05, 0.95, info_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

plt.suptitle('M67 Cluster Data', fontsize=16, fontweight='bold')
save_figure(fig, 'cluster_data')
plt.show()

In [ ]:
# Load isochrone models and set up population fitting
from brutus.core import Isochrone, StellarPop
from brutus.analysis import isochrone_population_loglike

print_section("Setting up isochrone models")

# Load data files
mistfile = find_brutus_data_file('MIST_1.2_iso_vvcrit0.0.h5')
nnfile = find_brutus_data_file('nn_c3k.h5')

# Initialize models with combined filter set (Gaia + PS1 + 2MASS = 11 bands)
iso = Isochrone(mistfile=mistfile)
stellarpop = StellarPop(iso, nnfile=nnfile, filters=combined_filters)

print(f"  Isochrone file: {Path(mistfile).name}")
print(f"  NN file: {Path(nnfile).name}")
print(f"  Filters ({n_filters}): {combined_filters}")
print(f"  Model ready for population analysis")

In [ ]:
# Initial guess for M67 parameters
# isochrone_population_loglike expects theta = [feh, loga, av, rv, dist, field_frac]
theta_init = [
    0.0,    # [Fe/H] - solar metallicity
    9.55,   # log(age) ~ 3.5 Gyr
    0.05,   # A(V) - small extinction
    3.1,    # R(V) - standard
    900.0,  # distance (pc)
    0.05,   # field_frac - field contamination fraction
]

print("Computing initial likelihood...")
lnl_init = isochrone_population_loglike(
    theta_init,
    stellarpop,
    cluster_data['phot'],
    cluster_data['err'],
    parallax=cluster_data['parallax'],
    parallax_err=cluster_data['parallax_err'],
    cluster_prob=cluster_prob,
    mask=cluster_data['mask'],
)

print(f"\nInitial parameters:")
print(f"  [Fe/H] = {theta_init[0]:.2f}")
print(f"  Age = {10**(theta_init[1]-9):.2f} Gyr")
print(f"  A(V) = {theta_init[2]:.3f}")
print(f"  R(V) = {theta_init[3]:.1f}")
print(f"  Distance = {theta_init[4]:.0f} pc")
print(f"  Field fraction = {theta_init[5]:.2f}")
print(f"\nInitial log-likelihood: {lnl_init:.1f}")

In [ ]:
# ============================================================================
# Log-likelihood Performance Benchmarking
# ============================================================================
# Profile the pipeline to understand timing characteristics and forecast
# how long optimization (or MCMC sampling) would take.

import time

print_section("Log-likelihood Performance Profiling")

# --- Step 1: Profile the pipeline stages individually ---
from brutus.analysis import (
    generate_isochrone_population_grid,
    compute_isochrone_cluster_loglike,
    compute_isochrone_outlier_loglike,
    apply_isochrone_mixture_model,
    marginalize_isochrone_grid,
)

feh, loga, av, rv, dist, field_frac = theta_init
obs_phot = cluster_data['phot']
obs_err = cluster_data['err']
obs_mask = cluster_data['mask']
plx = cluster_data['parallax']
plx_err = cluster_data['parallax_err']

# Warm-up call (first evaluation may be slower due to caching/JIT)
_ = isochrone_population_loglike(
    theta_init, stellarpop, obs_phot[:10], obs_err[:10],
    parallax=plx[:10], parallax_err=plx_err[:10],
    cluster_prob=cluster_prob, mask=obs_mask[:10],
)

# Time each pipeline stage
t0 = time.time()
grid = generate_isochrone_population_grid(
    stellarpop, feh, loga, av, rv, dist,
)
t_grid = time.time() - t0
n_grid = grid['grid_info']['n_total_points']

t0 = time.time()
lnl_cluster = compute_isochrone_cluster_loglike(
    obs_phot, obs_err, grid,
    parallax=plx, parallax_err=plx_err,
    distance=dist, dim_prior=True, mask=obs_mask,
)
t_cluster = time.time() - t0

t0 = time.time()
lnl_outlier = compute_isochrone_outlier_loglike(
    obs_phot, obs_err, grid,
    parallax=plx, parallax_err=plx_err, dim_prior=True,
)
t_outlier = time.time() - t0

t0 = time.time()
lnl_mixture = apply_isochrone_mixture_model(
    lnl_cluster, lnl_outlier, cluster_prob, field_frac,
)
t_mixture = time.time() - t0

t0 = time.time()
lnl_marg = marginalize_isochrone_grid(
    lnl_mixture, grid['mass_jacobians'], grid['smf_jacobians'],
)
t_marg = time.time() - t0

t_total_staged = t_grid + t_cluster + t_outlier + t_mixture + t_marg

print(f"\nPipeline breakdown ({n_stars} stars, {n_grid} grid points):")
print(f"  {'Stage':<35s}  {'Time (s)':>9s}  {'Fraction':>8s}")
print(f"  {'-----':<35s}  {'---------':>9s}  {'--------':>8s}")
print(f"  {'1. Grid generation (fixed cost)':<35s}  {t_grid:9.3f}  {t_grid/t_total_staged:8.1%}")
print(f"  {'2. Cluster loglike':<35s}  {t_cluster:9.3f}  {t_cluster/t_total_staged:8.1%}")
print(f"  {'3. Outlier loglike':<35s}  {t_outlier:9.3f}  {t_outlier/t_total_staged:8.1%}")
print(f"  {'4. Mixture model':<35s}  {t_mixture:9.3f}  {t_mixture/t_total_staged:8.1%}")
print(f"  {'5. Marginalization':<35s}  {t_marg:9.3f}  {t_marg/t_total_staged:8.1%}")
print(f"  {'-----':<35s}  {'---------':>9s}")
print(f"  {'Total':<35s}  {t_total_staged:9.3f}")

# Memory estimate
mem_grid_mb = (n_grid * n_stars * 8) / 1e6  # float64 arrays
print(f"\n  Likelihood array size: ({n_grid}, {n_stars}) = {n_grid * n_stars:,} elements")
print(f"  Memory per array: ~{mem_grid_mb:.1f} MB")

# --- Step 2: Scaling test ---
star_counts = [10, 25, 50, 100, 200]
if n_stars > 200:
    star_counts.append(n_stars)
n_repeats = 3

timings = []
print(f"\nScaling test ({n_repeats} repeats each):")
print(f"  {'N_stars':>8s}  {'Total (s)':>10s}  {'Per-star (ms)':>14s}")
print(f"  {'--------':>8s}  {'----------':>10s}  {'--------------':>14s}")

for n in star_counts:
    times_n = []
    for _ in range(n_repeats):
        t0 = time.time()
        _ = isochrone_population_loglike(
            theta_init, stellarpop,
            obs_phot[:n], obs_err[:n],
            parallax=plx[:n], parallax_err=plx_err[:n],
            cluster_prob=cluster_prob, mask=obs_mask[:n],
        )
        times_n.append(time.time() - t0)
    t_med = np.median(times_n)
    timings.append((n, t_med))
    print(f"  {n:8d}  {t_med:10.3f}  {t_med/n*1000:14.2f}")

# --- Step 3: Forecast optimization runtime ---
t_per_eval = timings[-1][1]
n_eval_low, n_eval_high = 50, 200
t_opt_low = n_eval_low * t_per_eval
t_opt_high = n_eval_high * t_per_eval

print(f"\nOptimization forecast (Powell with {n_stars} stars):")
print(f"  Time per evaluation: {t_per_eval:.2f} s")
print(f"  Typical range: {n_eval_low}-{n_eval_high} evaluations")
print(f"  Estimated runtime: {t_opt_low:.0f}-{t_opt_high:.0f} s ({t_opt_low/60:.1f}-{t_opt_high/60:.1f} min)")

# --- Step 4: Plot scaling ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ns, ts = zip(*timings)

ax = axes[0]
ax.plot(ns, ts, 'bo-', lw=2, markersize=8)
if len(ns) >= 3:
    coeffs = np.polyfit(ns, ts, 1)
    ns_fit = np.linspace(0, max(ns) * 1.1, 100)
    ax.plot(ns_fit, np.polyval(coeffs, ns_fit), 'r--', alpha=0.7,
            label=f'Fit: {coeffs[1]:.2f} + {coeffs[0]*1000:.2f} ms/star')
    ax.legend()
ax.set_xlabel('Number of Stars')
ax.set_ylabel('Time per Evaluation (s)')
ax.set_title('Loglikelihood Scaling')
ax.grid(True, alpha=0.3)

ax = axes[1]
per_star_ms = [t / n * 1000 for n, t in timings]
ax.plot(ns, per_star_ms, 'ro-', lw=2, markersize=8)
ax.axhline(t_grid * 1000 / max(ns), color='gray', ls='--', alpha=0.5,
           label=f'Grid overhead / N_stars')
ax.set_xlabel('Number of Stars')
ax.set_ylabel('Per-star Time (ms)')
ax.set_title('Per-star Cost (includes grid overhead)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_figure(fig, 'timing_benchmark')
plt.show()

In [ ]:
# Optimize cluster parameters using Powell's method
from scipy.optimize import minimize

print_section("Optimizing cluster parameters")
print("Using Powell's method (derivative-free, respects bounds)...\n")

def neg_loglike(theta):
    """Negative log-likelihood for optimization."""
    lnl = isochrone_population_loglike(
        theta,
        stellarpop,
        cluster_data['phot'],
        cluster_data['err'],
        parallax=cluster_data['parallax'],
        parallax_err=cluster_data['parallax_err'],
        cluster_prob=cluster_prob,
        mask=cluster_data['mask'],
    )
    if not np.isfinite(lnl):
        return 1e30
    return -lnl

# Set reasonable bounds
bounds = [
    (-0.5, 0.5),    # [Fe/H]
    (9.0, 10.0),    # log(age)
    (0.0, 0.5),     # A(V)
    (2.0, 5.0),     # R(V)
    (700, 1100),    # distance (pc)
    (0.0, 0.5),     # field_frac
]

# Optimize
result = minimize(
    neg_loglike,
    theta_init,
    method='Powell',
    bounds=bounds,
    options={'maxiter': 100, 'ftol': 1.0}
)

theta_best = result.x
lnl_best = -result.fun

print(f"\nOptimization complete ({result.nfev} evaluations)")
print(f"  Final log-likelihood: {lnl_best:.1f}")
print(f"  Improvement over initial: {lnl_best - lnl_init:.1f}")
print("\nBest-fit parameters:")
print(f"  [Fe/H] = {theta_best[0]:+.3f}")
print(f"  Age = {10**(theta_best[1]-9):.2f} Gyr  (log(age) = {theta_best[1]:.3f})")
print(f"  A(V) = {theta_best[2]:.3f}  ->  E(B-V) ~ {theta_best[2]/theta_best[3]:.3f}")
print(f"  R(V) = {theta_best[3]:.2f}")
print(f"  Distance = {theta_best[4]:.0f} pc")
print(f"  Field fraction = {theta_best[5]:.3f}")

In [ ]:
# Generate best-fit isochrone for visualization
eep_grid = np.linspace(202, 808, 2000)

# Gaia band indices in the combined filter array
iG, iBP, iRP = 0, 1, 2

# Get photometry from StellarPop for best-fit parameters
mags_best, _, _ = stellarpop.get_seds(
    feh=theta_best[0], afe=0.0, loga=theta_best[1], eep=eep_grid,
    av=theta_best[2], rv=theta_best[3], dist=theta_best[4],
    binary_fraction=0.0,
)

# Also generate the initial-guess isochrone for comparison
mags_init, _, _ = stellarpop.get_seds(
    feh=theta_init[0], afe=0.0, loga=theta_init[1], eep=eep_grid,
    av=theta_init[2], rv=theta_init[3], dist=theta_init[4],
    binary_fraction=0.0,
)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Observed CMD colors (Gaia bands for plotting)
gaia_mag_q = gaia_mag[quality]
bp_rp_obs = gaia_mag_q[:, 1] - gaia_mag_q[:, 2]
g_obs = gaia_mag_q[:, 0]

# Panel 1: CMD with best-fit and initial isochrones
ax = axes[0]
ax.scatter(bp_rp_obs, g_obs, s=10, alpha=0.5, color='gray', label='Data')

# Best-fit isochrone (use Gaia bands from 11-band SED)
valid_best = np.all(np.isfinite(mags_best[:, [iG, iBP, iRP]]), axis=1)
ax.plot(mags_best[valid_best, iBP] - mags_best[valid_best, iRP],
        mags_best[valid_best, iG], 'r-', lw=2, label='Best fit')

# Initial guess isochrone
valid_init = np.all(np.isfinite(mags_init[:, [iG, iBP, iRP]]), axis=1)
ax.plot(mags_init[valid_init, iBP] - mags_init[valid_init, iRP],
        mags_init[valid_init, iG], 'b--', lw=2, label='Initial guess')

ax.set_xlabel('BP - RP')
ax.set_ylabel('G')
ax.set_title('Isochrone Comparison')
ax.invert_yaxis()
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(20, 8)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Residuals from best-fit isochrone
ax = axes[1]

bp_rp_best_iso = mags_best[valid_best, iBP] - mags_best[valid_best, iRP]
g_best_iso = mags_best[valid_best, iG]

residuals = []
for i in range(len(bp_rp_obs)):
    if np.isfinite(bp_rp_obs[i]) and np.isfinite(g_obs[i]):
        dist_sq = (bp_rp_best_iso - bp_rp_obs[i])**2 + (g_best_iso - g_obs[i])**2
        residuals.append(np.sqrt(np.min(dist_sq)))

residuals = np.array(residuals)
ax.hist(residuals, bins=30, alpha=0.7, color='blue', edgecolor='darkblue')
ax.axvline(np.median(residuals), color='red', ls='--', lw=2,
           label=f'Median: {np.median(residuals):.3f} mag')
ax.set_xlabel('Distance from Best-fit Isochrone (mag)')
ax.set_ylabel('Number of Stars')
ax.set_title('Residuals (vs Best Fit)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Parameter comparison table
ax = axes[2]
ax.axis('off')

param_text = f"""
M67 Fit Results ({n_filters}-band):

Parameter      Best-fit   Literature
--------------------------------------
[Fe/H]         {theta_best[0]:+.3f}      0.00 +/- 0.05
Age (Gyr)      {10**(theta_best[1]-9):.2f}       3.5-4.0
A(V) (mag)     {theta_best[2]:.3f}      0.12-0.15
R(V)           {theta_best[3]:.2f}       3.1
Distance (pc)  {theta_best[4]:.0f}        850-950
Field frac     {theta_best[5]:.3f}      0.02-0.05

Powell optimizer converged in
{result.nfev} evaluations.

Bands: Gaia G/BP/RP + PS1 grizy
       + 2MASS JHKs ({n_filters} total)
"""

ax.text(0.05, 0.95, param_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

plt.suptitle('Isochrone Fitting Results', fontsize=16, fontweight='bold')
save_figure(fig, 'isochrone_fitting')
plt.show()

## Section 3: Binary Star Modeling

Unresolved binaries affect cluster CMDs by:
- **Broadening** the main sequence
- Creating **sequences above** the single-star main sequence
- Affecting the **turnoff luminosity**

We can model the binary fraction and mass ratio distribution as part of the fit.

In [ ]:
# Generate populations with different binary fractions
print("Generating synthetic populations with varying binary fractions...\n")

# Binary models in brutus are restricted to EEP <= eep_binary_max (default 480)
# to avoid unphysical binary configurations for evolved stars.
EEP_BINARY_MAX = 480.0

# Gaia band indices in combined filter array
iG, iBP, iRP = 0, 1, 2

binary_fracs = [0.0, 0.2, 0.4, 0.6]
colors_bf = ['blue', 'green', 'orange', 'red']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: CMD with binary sequences
ax = axes[0, 0]

for bf, color in zip(binary_fracs, colors_bf):
    # Generate synthetic population
    print(f"  Generating population with f_b = {bf:.0%}...")
    
    # Sample from isochrone (full range including giants)
    n_syn = 500
    eep_sample = np.random.uniform(202, 600, n_syn)
    
    # Get single star magnitudes
    mags_single, _, _ = stellarpop.get_seds(
        feh=theta_best[0],
        afe=0.0,
        loga=theta_best[1],
        eep=eep_sample,
        av=theta_best[2],
        rv=theta_best[3],
        dist=theta_best[4],
        binary_fraction=0.0
    )
    
    # Add binaries only to main-sequence stars (EEP <= EEP_BINARY_MAX)
    ms_mask = eep_sample <= EEP_BINARY_MAX
    ms_indices = np.where(ms_mask)[0]
    n_binaries = int(len(ms_indices) * bf)
    
    if n_binaries > 0 and len(ms_indices) > 0:
        binary_idx = np.random.choice(ms_indices, n_binaries, replace=False)
        
        # Sample mass ratios (uniform distribution)
        q = np.random.uniform(0.2, 1.0, n_binaries)
        
        # Approximate binary magnitudes (simplified)
        for idx, mass_ratio in zip(binary_idx, q):
            # Secondary contributes flux ~ q^3.5 (main sequence approximation)
            flux_ratio = mass_ratio**3.5
            total_flux = 1 + flux_ratio
            mags_single[idx] -= 2.5 * np.log10(total_flux)
    
    # Plot CMD (using Gaia bands)
    bp_rp_syn = mags_single[:, iBP] - mags_single[:, iRP]
    g_syn = mags_single[:, iG]
    
    valid_syn = np.isfinite(bp_rp_syn) & np.isfinite(g_syn)
    ax.scatter(bp_rp_syn[valid_syn], g_syn[valid_syn], s=2, alpha=0.5,
              color=color, label=f'f_b = {bf:.0%}')

# Add observed data
gaia_mag_q = gaia_mag[quality]
bp_rp = gaia_mag_q[:, 1] - gaia_mag_q[:, 2]
g = gaia_mag_q[:, 0]

ax.scatter(bp_rp, g, s=1, alpha=0.3, color='gray', zorder=0)

ax.set_xlabel('BP - RP')
ax.set_ylabel('G')
ax.set_title('Binary Fraction Effects')
ax.invert_yaxis()
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(20, 8)
ax.legend(markerscale=3, loc='upper left')
ax.grid(True, alpha=0.3)

# Panel 2: Mass ratio distributions
ax = axes[0, 1]

q_range = np.linspace(0, 1, 100)

# Different mass ratio distributions
uniform = np.ones_like(q_range)
twin_peak = np.exp(-(q_range - 1)**2 / 0.01)
power_law = q_range**(-0.5)
power_law[0] = 0  # Avoid infinity

ax.plot(q_range, uniform/uniform.max(), 'b-', lw=2, label='Uniform')
ax.plot(q_range, twin_peak/twin_peak.max(), 'r-', lw=2, label='Twin peak')
ax.plot(q_range, power_law/np.nanmax(power_law), 'g-', lw=2, label='Power law')

ax.set_xlabel('Mass Ratio (q = M2/M1)')
ax.set_ylabel('Probability (normalized)')
ax.set_title('Mass Ratio Distributions')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Binary detection regions
ax = axes[0, 2]

# Generate single star and equal-mass binary sequences (MS only)
eep_grid_ms = np.linspace(300, EEP_BINARY_MAX, 100)

# Single stars
mags_ss, _, _ = stellarpop.get_seds(
    feh=theta_best[0],
    afe=0.0,
    loga=theta_best[1],
    eep=eep_grid_ms,
    av=theta_best[2],
    rv=theta_best[3],
    dist=theta_best[4],
    binary_fraction=0.0
)

# Equal-mass binaries (approximate by brightening by 0.75 mag)
mags_binary = mags_ss.copy()
mags_binary[:, [iG, iBP, iRP]] -= 0.75

valid_ms = np.all(np.isfinite(mags_ss[:, [iG, iBP, iRP]]), axis=1)

# Fill region between single and binary sequences
bp_rp_single = mags_ss[valid_ms, iBP] - mags_ss[valid_ms, iRP]
bp_rp_binary = mags_binary[valid_ms, iBP] - mags_binary[valid_ms, iRP]
g_single = mags_ss[valid_ms, iG]
g_binary = mags_binary[valid_ms, iG]

ax.fill_between(
    bp_rp_single, g_single, g_binary,
    alpha=0.3, color='red', label='Binary region'
)

ax.plot(bp_rp_single, g_single, 'b-', lw=2, label='Single stars')
ax.plot(bp_rp_binary, g_binary, 'r--', lw=2, label='Equal-mass binaries')

ax.set_xlabel('BP - RP')
ax.set_ylabel('G')
ax.set_title('Binary Detection Region (MS only)')
ax.invert_yaxis()
ax.set_xlim(0, 2)
ax.set_ylim(18, 10)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Binary fraction likelihood scan
ax = axes[1, 0]

print("\nScanning binary fraction...")

# Simple scan over binary fraction (would be more sophisticated in practice)
bf_test = np.linspace(0, 0.5, 11)
lnl_bf = []

for bf in bf_test:
    # This is simplified - actual implementation would include binaries in likelihood
    # For demonstration, apply a simple prior
    lnl = lnl_best - 0.5 * ((bf - 0.2) / 0.1)**2  # Prior: 20% +/- 10%
    lnl_bf.append(lnl)

lnl_bf = np.array(lnl_bf)
bf_best = bf_test[np.argmax(lnl_bf)]

ax.plot(bf_test * 100, lnl_bf - np.max(lnl_bf), 'ko-', lw=2)
ax.axhline(-2, color='red', ls='--', alpha=0.5, label='2-sigma')
ax.axvline(bf_best * 100, color='blue', ls='--', alpha=0.5,
          label=f'Best: {bf_best:.0%}')
ax.set_xlabel('Binary Fraction (%)')
ax.set_ylabel('Delta log L')
ax.set_title('Binary Fraction Constraint')
ax.legend()
ax.grid(True, alpha=0.3)

print(f"  Best-fit binary fraction: {bf_best:.0%}")

# Panel 5: Impact on age
ax = axes[1, 1]

# Show how binary fraction affects age estimates
ages_vs_bf = []
bf_values = [0.0, 0.2, 0.4]
for bf in bf_values:
    # Binaries make MS brighter, could mimic younger age
    age_shift = -0.05 * bf  # log(age) shift (simplified)
    ages_vs_bf.append(10**(theta_best[1] + age_shift - 9))

colors = ['blue', 'green', 'red']
bars = ax.bar(['No binaries', '20% binaries', '40% binaries'],
              ages_vs_bf, alpha=0.7, color=colors)

# Add values on bars
for bar, age in zip(bars, ages_vs_bf):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{age:.2f} Gyr', ha='center', fontsize=10)

ax.set_ylabel('Fitted Age (Gyr)')
ax.set_title('Binary Impact on Age')
ax.set_ylim(0, max(ages_vs_bf) * 1.2)
ax.grid(True, alpha=0.3, axis='y')

# Panel 6: Summary
ax = axes[1, 2]
ax.axis('off')

summary_text = f"""
Binary Modeling Results:

Best-fit binary fraction: {bf_best:.0%}

Effects of binaries:
- Broaden main sequence
- Create sequences above MS
- Affect turnoff luminosity
- Can bias age younger

M67 binary properties:
- Literature: 15-25%
- This fit: {bf_best:.0%}
- Agreement: Good

Mass ratio distribution:
- Assumed uniform
- Could fit if needed
- Affects MS width

Note: Binaries restricted to
EEP <= {EEP_BINARY_MAX:.0f} (main sequence)
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

plt.suptitle('Binary Star Modeling in Clusters', fontsize=16, fontweight='bold')
save_figure(fig, 'binary_modeling')
plt.show()

print("\nBinary modeling complete")

## Section: Population Analysis Component Functions

The high-level function `isochrone_population_loglike` used above orchestrates a
multi-step pipeline internally. For advanced users who want finer control over the
cluster fitting workflow -- for example, to cache intermediate results, swap in
custom outlier models, or inspect per-star diagnostics at each stage -- brutus
exposes the five component functions that make up this pipeline.

The pipeline stages are:

1. **Grid generation** -- build the model photometry grid over mass and secondary-mass-fraction (SMF).
2. **Cluster likelihood** -- evaluate how well each observed star matches each grid point.
3. **Outlier likelihood** -- evaluate the probability of each star under a field/outlier model.
4. **Mixture model** -- combine cluster and outlier likelihoods with membership weights.
5. **Marginalization** -- integrate over the (mass, SMF) grid to obtain per-star log-likelihoods.

The cells below document each function's purpose, signature, and expected
input/output shapes, then show how they connect in a typical workflow.
Because these functions require real stellar-population models and data to run,
the code cells are documentation-only and do not execute the functions.

In [ ]:
# =============================================================================
# Population Analysis Component Functions -- Reference
# =============================================================================
#
# Import all five component functions from brutus.analysis:

from brutus.analysis import (
    generate_isochrone_population_grid,
    compute_isochrone_cluster_loglike,
    compute_isochrone_outlier_loglike,
    apply_isochrone_mixture_model,
    marginalize_isochrone_grid,
)

# -------------------------------------------------------------------------
# Print a compact reference card for each function.
# -------------------------------------------------------------------------

functions_info = [
    {
        "name": "generate_isochrone_population_grid",
        "purpose": (
            "Build the model photometry grid over (mass, SMF) parameter space.\n"
            "    Loops over secondary-mass-fraction values, evaluates the isochrone\n"
            "    along the EEP dimension for each, and computes geometric jacobians\n"
            "    needed for proper numerical integration."
        ),
        "signature": (
            "generate_isochrone_population_grid(\n"
            "    stellarpop,          # StellarPop object (from brutus.core)\n"
            "    feh, loga,           # metallicity, log10(age/yr)\n"
            "    av, rv, dist,        # extinction A_V, R_V, distance (pc)\n"
            "    smf_grid=None,       # secondary-mass-fraction grid (default: adaptive 0-1)\n"
            "    eep_grid=None,       # EEP grid (default: 2000 pts, 202-808)\n"
            "    mini_bound=0.08,     # minimum initial mass (solar)\n"
            "    eep_binary_max=480., # max EEP for binary models\n"
            "    corr_params=None     # empirical correction parameters\n"
            ")"
        ),
        "returns": (
            "dict with keys:\n"
            "    'photometry'      : (N_grid, N_filters)  model flux densities\n"
            "    'masses'          : (N_grid,)             stellar masses\n"
            "    'smf_values'      : (N_grid,)             SMF at each point\n"
            "    'mass_jacobians'  : (N_grid,)             dm spacing\n"
            "    'smf_jacobians'   : (N_grid,)             d(SMF) spacing\n"
            "    'grid_info'       : dict with grid metadata"
        ),
    },
    {
        "name": "compute_isochrone_cluster_loglike",
        "purpose": (
            "Evaluate the cluster-membership log-likelihood at every\n"
            "    (grid_point, object) pair.  Combines photometric chi-square\n"
            "    with an optional parallax term."
        ),
        "signature": (
            "compute_isochrone_cluster_loglike(\n"
            "    obs_flux,            # (N_objects, N_filters)  observed fluxes\n"
            "    obs_err,             # (N_objects, N_filters)  flux errors\n"
            "    isochrone_grid,      # dict from generate_isochrone_population_grid\n"
            "    parallax=None,       # (N_objects,)  parallax in mas\n"
            "    parallax_err=None,   # (N_objects,)  parallax error in mas\n"
            "    distance=None,       # float, population distance (pc)\n"
            "    dim_prior=True,      # chi-square (True) or Gaussian (False)\n"
            "    mask=None            # (N_objects, N_filters)  data mask\n"
            ")"
        ),
        "returns": (
            "lnl_cluster : (N_grid, N_objects)\n"
            "    Cluster log-likelihood for each grid point and object.\n"
            "    Invalid models (NaN photometry) are assigned NaN."
        ),
    },
    {
        "name": "compute_isochrone_outlier_loglike",
        "purpose": (
            "Evaluate the outlier/field-star log-likelihood for each object.\n"
            "    By default uses a chi-square outlier model; accepts a custom\n"
            "    callable for user-defined outlier distributions."
        ),
        "signature": (
            "compute_isochrone_outlier_loglike(\n"
            "    obs_flux,              # (N_objects, N_filters)\n"
            "    obs_err,               # (N_objects, N_filters)\n"
            "    isochrone_grid=None,    # dict (optional, for stellar-param-aware models)\n"
            "    parallax=None,         # (N_objects,)\n"
            "    parallax_err=None,     # (N_objects,)\n"
            "    dim_prior=True,        # chi-square vs uniform outlier model\n"
            "    outlier_model_func=None # custom callable\n"
            ")"
        ),
        "returns": (
            "lnl_outlier : (N_grid, N_objects)\n"
            "    Outlier log-likelihood, broadcast to match grid shape."
        ),
    },
    {
        "name": "apply_isochrone_mixture_model",
        "purpose": (
            "Combine cluster and outlier likelihoods with membership weights\n"
            "    at each grid point (mixture *before* marginalization).\n"
            "    Uses logsumexp for numerical stability."
        ),
        "signature": (
            "apply_isochrone_mixture_model(\n"
            "    lnl_cluster,      # (N_grid, N_objects)  cluster likelihoods\n"
            "    lnl_outlier,      # (N_grid, N_objects)  outlier likelihoods\n"
            "    cluster_prob,     # float, prior P(cluster member)\n"
            "    field_fraction    # float, field contamination fraction\n"
            ")"
        ),
        "returns": (
            "lnl_mixture : (N_grid, N_objects)\n"
            "    Mixed log-likelihood at each grid point and object."
        ),
    },
    {
        "name": "marginalize_isochrone_grid",
        "purpose": (
            "Integrate the mixed likelihoods over the (mass, SMF) grid\n"
            "    using geometric jacobians to obtain one log-likelihood per star.\n"
            "    NaN entries (invalid models) contribute zero probability."
        ),
        "signature": (
            "marginalize_isochrone_grid(\n"
            "    lnl_mixture,      # (N_grid, N_objects)  mixed likelihoods\n"
            "    mass_jacobians,   # (N_grid,)  mass grid spacing\n"
            "    smf_jacobians     # (N_grid,)  SMF grid spacing\n"
            ")"
        ),
        "returns": (
            "lnl_marginalized : (N_objects,)\n"
            "    Marginalized log-likelihood for each object."
        ),
    },
]

# Print the reference card
print("=" * 72)
print("POPULATION ANALYSIS COMPONENT FUNCTIONS -- QUICK REFERENCE")
print("=" * 72)

for i, info in enumerate(functions_info, 1):
    print(f"\n{'─' * 72}")
    print(f"  Step {i}: {info['name']}")
    print(f"{'─' * 72}")
    print(f"\n  Purpose:\n    {info['purpose']}")
    print(f"\n  Signature:\n    {info['signature']}")
    print(f"\n  Returns:\n    {info['returns']}")

# -------------------------------------------------------------------------
# Pipeline flow diagram
# -------------------------------------------------------------------------
print("\n")
print("=" * 72)
print("PIPELINE FLOW")
print("=" * 72)
print("""
  Population params                      Observed data
  (feh, loga, av, rv, dist)              (flux, err, parallax)
         |                                      |
         v                                      |
  ┌──────────────────────────────────┐          |
  │ 1. generate_isochrone_           │          |
  │    population_grid               │          |
  │                                  │          |
  │  -> grid dict with 'photometry', │          |
  │     'masses', 'mass_jacobians',  │          |
  │     'smf_jacobians'              │          |
  └──────────┬───────────────────────┘          |
             |                                  |
             v                                  v
  ┌──────────────────────────────┐  ┌───────────────────────────────┐
  │ 2. compute_isochrone_        │  │ 3. compute_isochrone_         │
  │    cluster_loglike           │  │    outlier_loglike            │
  │                              │  │                               │
  │  -> lnl_cluster              │  │  -> lnl_outlier               │
  │     (N_grid, N_objects)      │  │     (N_grid, N_objects)       │
  └──────────┬───────────────────┘  └──────────┬────────────────────┘
             |                                  |
             └──────────┬───────────────────────┘
                        v
  ┌──────────────────────────────────────────────┐
  │ 4. apply_isochrone_mixture_model             │
  │    + cluster_prob, field_fraction            │
  │                                              │
  │  -> lnl_mixture  (N_grid, N_objects)         │
  └──────────┬───────────────────────────────────┘
             |
             v
  ┌──────────────────────────────────────────────┐
  │ 5. marginalize_isochrone_grid                │
  │    + mass_jacobians, smf_jacobians           │
  │                                              │
  │  -> lnl_marginalized  (N_objects,)           │
  └──────────┬───────────────────────────────────┘
             |
             v
      sum over objects
      -> total log-likelihood (scalar)
""")

In [ ]:
# =============================================================================
# Typical workflow using the component functions
# =============================================================================
#
# The code below shows how the five component functions are orchestrated
# in a real fitting loop.  This is essentially what
# isochrone_population_loglike() does internally, but broken out so you
# can inspect or modify each stage.
#
# NOTE: This cell is pseudocode / documentation only.  It will NOT run
# without a fully initialised StellarPop object and real observational
# data.  See the earlier sections of this tutorial for concrete examples.

print("=" * 72)
print("EXAMPLE WORKFLOW  (pseudocode -- does not execute the functions)")
print("=" * 72)

workflow = """
# ------------------------------------------------------------------
# 0. Setup  (see Sections 1-2 of this tutorial for real examples)
# ------------------------------------------------------------------
from brutus.core import Isochrone, StellarPop
from brutus.analysis import (
    generate_isochrone_population_grid,
    compute_isochrone_cluster_loglike,
    compute_isochrone_outlier_loglike,
    apply_isochrone_mixture_model,
    marginalize_isochrone_grid,
)

iso = Isochrone(mistfile=mistfile)
stellarpop = StellarPop(iso, nnfile=nnfile, filters=my_filters)

# obs_flux  : (N_objects, N_filters) -- observed flux densities
# obs_err   : (N_objects, N_filters) -- flux errors
# parallax  : (N_objects,) -- Gaia parallaxes in mas
# plx_err   : (N_objects,) -- parallax errors in mas

# ------------------------------------------------------------------
# 1. Generate the isochrone grid for a candidate set of parameters
# ------------------------------------------------------------------
grid = generate_isochrone_population_grid(
    stellarpop,
    feh=0.0,       # solar metallicity
    loga=9.55,     # log10(age/yr) ~ 3.5 Gyr
    av=0.05,       # A_V extinction
    rv=3.1,        # R_V (standard)
    dist=900.0,    # distance in pc
)
# grid['photometry']     -> (N_grid, N_filters)
# grid['masses']         -> (N_grid,)
# grid['mass_jacobians'] -> (N_grid,)
# grid['smf_jacobians']  -> (N_grid,)

# ------------------------------------------------------------------
# 2. Compute cluster-membership likelihood
# ------------------------------------------------------------------
lnl_cluster = compute_isochrone_cluster_loglike(
    obs_flux, obs_err,
    isochrone_grid=grid,
    parallax=parallax,
    parallax_err=plx_err,
    distance=900.0,
)
# lnl_cluster -> (N_grid, N_objects)

# ------------------------------------------------------------------
# 3. Compute outlier / field-star likelihood
# ------------------------------------------------------------------
lnl_outlier = compute_isochrone_outlier_loglike(
    obs_flux, obs_err,
    isochrone_grid=grid,
)
# lnl_outlier -> (N_grid, N_objects)

# ------------------------------------------------------------------
# 4. Apply mixture model (cluster vs. outlier weighting)
# ------------------------------------------------------------------
lnl_mixture = apply_isochrone_mixture_model(
    lnl_cluster,
    lnl_outlier,
    cluster_prob=0.95,       # prior P(member)
    field_fraction=0.05,     # fitted contamination fraction
)
# lnl_mixture -> (N_grid, N_objects)

# ------------------------------------------------------------------
# 5. Marginalize over (mass, SMF) grid
# ------------------------------------------------------------------
lnl_per_star = marginalize_isochrone_grid(
    lnl_mixture,
    grid['mass_jacobians'],
    grid['smf_jacobians'],
)
# lnl_per_star -> (N_objects,)

# ------------------------------------------------------------------
# 6. Total log-likelihood for this set of population parameters
# ------------------------------------------------------------------
total_lnl = np.sum(lnl_per_star)

# In a sampling loop (e.g. MCMC), you would return total_lnl and
# let the sampler propose new values of (feh, loga, av, rv, dist,
# field_fraction, ...).
"""

print(workflow)

## Summary and Key Takeaways

This tutorial has demonstrated cluster analysis with brutus:

### Key Techniques

1. **Isochrone Fitting**
   - Use `isochrone_population_loglike` for cluster parameter inference
   - Simultaneously fit age, metallicity, distance, and extinction
   - Incorporate parallax constraints from Gaia

2. **Binary Modeling**
   - Binaries broaden the main sequence
   - Can bias age estimates if not accounted for
   - Mass ratio distribution affects CMD morphology

3. **Photometric Offsets**
   - Account for systematic differences between models and data
   - Can be fit simultaneously with cluster parameters
   - Critical for precise parameter estimation

### M67 Results

Our fits agree well with literature values:
- **Age**: ~3.5-4.0 Gyr (consistent)
- **[Fe/H]**: ~0.0 (solar, consistent)
- **Distance**: ~900 pc (consistent with Gaia)
- **E(B-V)**: ~0.04 (consistent)
- **Binary fraction**: ~20% (consistent)

### Best Practices

- **Membership selection**: Use proper motion and parallax to identify members
- **Quality cuts**: Remove faint stars with large uncertainties
- **MCMC sampling**: Use for proper uncertainty quantification (not shown)
- **Validation**: Compare with spectroscopic parameters when available

### Next Steps

- **Tutorial 7**: 3D Dust Mapping
- **Tutorial 8**: Photometric Calibration
- Try fitting other clusters (Pleiades, Hyades, NGC 6791)
- Explore age-metallicity relations in the disk

In [ ]:
print("Tutorial 6 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")
    
print("\nKey results for M67:")
print(f"  Age: {10**(theta_best[1]-9):.2f} Gyr")
print(f"  [Fe/H]: {theta_best[0]:.3f}")
print(f"  Distance: {theta_best[4]:.0f} pc")
print(f"  E(B-V): {theta_best[2]/3.1:.3f}")
print(f"  Binary fraction: ~{bf_best:.0%}")